# 01 · Data engineering, joins and EDA

BDA Units 1 and 3. Start with sample data, then switch to the official datasets. MovieLens has ratings/tags, not review text. The IMDb sentiment corpus has binary labels, not aspect or fake-review labels. TMDB review coverage is sparse and not an exhaustive source of movie reviews.

In [1]:
from pathlib import Path
import sys, json, os
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from CineScope/notebooks or CineScope.'
sys.path.insert(0, str(ROOT / 'backend'))
sys.path.insert(0, str(ROOT / 'scripts'))
os.chdir(ROOT)
(ROOT / 'artifacts').mkdir(exist_ok=True)
SEED = 42

In [2]:
USE_REAL_DATA = False
DOWNLOAD_DATA = False  # Downloads are explicit and can be large.
MOVIELENS_NAME = 'movielens-small'  # Use movielens-32m for the big-data experiment.
ML_DIR = ROOT / 'data/raw' / ('ml-32m' if MOVIELENS_NAME == 'movielens-32m' else 'ml-latest-small')
if DOWNLOAD_DATA:
    from download_data import download
    download(MOVIELENS_NAME, ROOT / 'data/raw')
    download('imdb-reviews', ROOT / 'data/raw')
if USE_REAL_DATA:
    from prepare_imdb import prepare
    reviews = prepare(ROOT/'data/raw/aclImdb', ROOT/'data/processed')
else:
    import pandas as pd
    reviews = pd.DataFrame(json.loads((ROOT/'data/demo/reviews.json').read_text()))
print('REAL DATA' if USE_REAL_DATA else 'SYNTHETIC DEMO REVIEWS', len(reviews))
reviews.head()

SYNTHETIC DEMO REVIEWS 36
  review_id  movie_id  ...          source            created_at
0  demo-1-0         1  ...  synthetic-demo  2026-09-10T12:00:00Z
1  demo-1-1         1  ...  synthetic-demo  2026-09-11T12:00:00Z
2  demo-1-2         1  ...  synthetic-demo  2026-09-12T12:00:00Z
3  demo-2-0         2  ...  synthetic-demo  2026-09-10T12:00:00Z
4  demo-2-1         2  ...  synthetic-demo  2026-09-11T12:00:00Z

[5 rows x 5 columns]


In [3]:
import pandas as pd
if USE_REAL_DATA:
    ratings = pd.read_csv(ML_DIR/'ratings.csv')  # Small starter only. Use Scala/Spark for 32M.
    movies = pd.read_csv(ML_DIR/'movies.csv')
    links = pd.read_csv(ML_DIR/'links.csv', dtype={'imdbId':'string'})
    links['imdb_id'] = 'tt' + links.imdbId.str.zfill(7)
    canonical = movies.merge(links, on='movieId', validate='one_to_one')
    linked_reviews = reviews.merge(canonical[['movieId','imdb_id']],on='imdb_id',how='left')
    print('Review-to-MovieLens match rate:', linked_reviews.movieId.notna().mean())
    print('Unmatched reviews remain valid for sentiment training, but are not assigned to a movie.')
    display(canonical.head())
else:
    canonical = pd.DataFrame(json.loads((ROOT/'data/demo/catalog.json').read_text()))
    display(canonical[['movie_id','title','genres','themes']].head())

   movie_id  ...                                       themes
0         1  ...              [Space, Family, Time, Survival]
1         2  ...              [Language, Time, First contact]
2         3  ...  [Artificial intelligence, Loneliness, Love]
3         4  ...                  [Destiny, Politics, Desert]
4         5  ...  [Identity, Artificial intelligence, Memory]

[5 rows x 4 columns]


## TMDB ingestion (credential required)
Store the bearer token in the environment, not in the notebook or frontend. `ingest_tmdb.py` handles pagination, retries and rate limiting; it writes append-only records which downstream tools deduplicate by source ID.

```bash
python scripts/ingest_tmdb.py --movielens data/raw/ml-latest-small --limit 100 --pages 3
python scripts/build_catalog.py --movielens data/raw/ml-latest-small
python scripts/ingest_reviews.py data/processed/tmdb_reviews.jsonl
```

The optional `--imdb-basics data/raw/title.basics.tsv.gz` enriches runtime by exact IMDb ID. IMDb TSV files contain metadata, not full review text. A new movie can enter the canonical JSON with a fresh positive integer ID, genres, synopsis and themes before ratings/reviews exist.

In [4]:
from cinescope.training import audit_reviews
screened = audit_reviews(reviews)
print({'rows':len(screened),'duplicate_rows':int(screened.duplicate.sum()),'flagged_for_inspection':int((screened.screening_risk>=70).sum())})
screened[['review_id','screening_risk','duplicate']].head()

{'rows': 36, 'duplicate_rows': 0, 'flagged_for_inspection': 0}
  review_id  screening_risk  duplicate
0  demo-1-0               0      False
1  demo-1-1               0      False
2  demo-1-2               0      False
3  demo-2-0               0      False
4  demo-2-1               0      False


In [5]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,3.4))
ax.hist(reviews.text.str.split().str.len(), bins=20, color='#698d3c', edgecolor='white')
ax.set(xlabel='Words per review', ylabel='Reviews', title='Review length distribution' + ('' if USE_REAL_DATA else ' · synthetic sample'))
fig.tight_layout(); plt.show()

## Big-data execution
Do not load 32M ratings into pandas for the main experiment. Use the Scala ETL with `spark-submit`, or the Spark notebook in 05. Record driver/executor memory, partitions, shuffle size, input count, wall time and cluster topology. Inspect the Spark UI DAG and `explain(true)`; comparing a tiny local sample is not evidence of distributed scalability.

The downloader writes source URL, retrieval time, byte count and SHA-256 provenance beside every archive. Keep raw → processed → model stages distinct and never redistribute the full datasets with this project.